<a href="https://colab.research.google.com/github/pranathi-hem/paperback-gpt-dpo/blob/main/runwith13million.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
checkpoint_dir = '/content/drive/MyDrive/nanogpt_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

Mounted at /content/drive


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True
Tesla T4


In [3]:
!rm -rf ~/.cache/huggingface/datasets

In [4]:
# =========================================================
# Expanded dataset - 25 public-domain novels instead of 5.
# Same direct-download approach, no Hugging Face involved.
# Bigger dataset supports longer pretraining (5000+ iters)
# without overfitting as fast.
# =========================================================

import urllib.request

urls = [
    "https://www.gutenberg.org/files/1342/1342-0.txt",   # Pride and Prejudice - Austen
    "https://www.gutenberg.org/files/11/11-0.txt",        # Alice's Adventures in Wonderland - Carroll
    "https://www.gutenberg.org/files/84/84-0.txt",        # Frankenstein - Shelley
    "https://www.gutenberg.org/files/2701/2701-0.txt",    # Moby-Dick - Melville
    "https://www.gutenberg.org/files/98/98-0.txt",        # A Tale of Two Cities - Dickens
    "https://www.gutenberg.org/files/1400/1400-0.txt",    # Great Expectations - Dickens
    "https://www.gutenberg.org/files/76/76-0.txt",        # Adventures of Huckleberry Finn - Twain
    "https://www.gutenberg.org/files/74/74-0.txt",        # The Adventures of Tom Sawyer - Twain
    "https://www.gutenberg.org/files/345/345-0.txt",      # Dracula - Stoker
    "https://www.gutenberg.org/files/1260/1260-0.txt",    # Jane Eyre - Bronte
    "https://www.gutenberg.org/files/768/768-0.txt",      # Wuthering Heights - Bronte
    "https://www.gutenberg.org/files/55/55-0.txt",        # The Wonderful Wizard of Oz - Baum
    "https://www.gutenberg.org/files/120/120-0.txt",      # Treasure Island - Stevenson
    "https://www.gutenberg.org/files/43/43-0.txt",        # Dr. Jekyll and Mr. Hyde - Stevenson
    "https://www.gutenberg.org/files/174/174-0.txt",      # The Picture of Dorian Gray - Wilde
    "https://www.gutenberg.org/files/2591/2591-0.txt",    # Grimm's Fairy Tales
    "https://www.gutenberg.org/files/1661/1661-0.txt",    # The Adventures of Sherlock Holmes - Doyle
    "https://www.gutenberg.org/files/219/219-0.txt",      # Heart of Darkness - Conrad
    "https://www.gutenberg.org/files/5200/5200-0.txt",    # Metamorphosis - Kafka
    "https://www.gutenberg.org/files/158/158-0.txt",      # Emma - Austen
    "https://www.gutenberg.org/files/161/161-0.txt",      # Sense and Sensibility - Austen
    "https://www.gutenberg.org/files/36/36-0.txt",        # The War of the Worlds - Wells
    "https://www.gutenberg.org/files/35/35-0.txt",        # The Time Machine - Wells
    "https://www.gutenberg.org/files/1952/1952-0.txt",    # The Yellow Wallpaper - Gilman
    "https://www.gutenberg.org/files/6130/6130-0.txt",    # The Iliad - Homer (trans. Butler)
]

texts = []
for u in urls:
    try:
        local_path, _ = urllib.request.urlretrieve(u)
        with open(local_path, "r", encoding="utf-8") as f:
            texts.append(f.read())
        print(f"loaded: {u}")
    except Exception as e:
        print(f"FAILED: {u} -> {e}")

text = "\n".join(texts)
print(f"\ndataset length: {len(text)} characters ({len(texts)}/{len(urls)} books loaded)")

loaded: https://www.gutenberg.org/files/1342/1342-0.txt
loaded: https://www.gutenberg.org/files/11/11-0.txt
loaded: https://www.gutenberg.org/files/84/84-0.txt
loaded: https://www.gutenberg.org/files/2701/2701-0.txt
loaded: https://www.gutenberg.org/files/98/98-0.txt
loaded: https://www.gutenberg.org/files/1400/1400-0.txt
loaded: https://www.gutenberg.org/files/76/76-0.txt
loaded: https://www.gutenberg.org/files/74/74-0.txt
loaded: https://www.gutenberg.org/files/345/345-0.txt
loaded: https://www.gutenberg.org/files/1260/1260-0.txt
loaded: https://www.gutenberg.org/files/768/768-0.txt
loaded: https://www.gutenberg.org/files/55/55-0.txt
loaded: https://www.gutenberg.org/files/120/120-0.txt
loaded: https://www.gutenberg.org/files/43/43-0.txt
loaded: https://www.gutenberg.org/files/174/174-0.txt
loaded: https://www.gutenberg.org/files/2591/2591-0.txt
loaded: https://www.gutenberg.org/files/1661/1661-0.txt
loaded: https://www.gutenberg.org/files/219/219-0.txt
loaded: https://www.gutenberg.

In [5]:
import torch
!pip install tiktoken --quiet

import tiktoken
enc = tiktoken.get_encoding("gpt2")

vocab_size = enc.n_vocab   # ~50257
data_ids = enc.encode(text)
data = torch.tensor(data_ids, dtype=torch.long)

encode = enc.encode
decode = enc.decode

In [6]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [7]:
import torch
block_size=128
batch_size=100

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y



In [8]:
n_embd = 256
no_heads = 8
n_layer = 8
max_len=2*block_size
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [9]:

def precompute_rope_freqs(head_size, max_seq_len, device, theta=10000.0):
    # head_size must be even — we rotate dimensions in pairs
    inv_freq = 1.0 / (theta ** (torch.arange(0, head_size, 2, device=device).float() / head_size))
    positions = torch.arange(max_seq_len, device=device).float()
    freqs = torch.outer(positions, inv_freq)  # (max_seq_len, head_size/2)
    cos = freqs.cos()
    sin = freqs.sin()
    return cos, sin  # each: (max_seq_len, head_size/2)

def apply_rope(x, cos, sin):
    # x: (B, T, head_size)
    T = x.shape[1]
    cos = cos[:T].unsqueeze(0)  # (1, T, head_size/2)
    sin = sin[:T].unsqueeze(0)  # (1, T, head_size/2)

    x1 = x[..., 0::2]  # even dims
    x2 = x[..., 1::2]  # odd dims

    rotated_x1 = x1 * cos - x2 * sin
    rotated_x2 = x1 * sin + x2 * cos

    out = torch.empty_like(x)
    out[..., 0::2] = rotated_x1
    out[..., 1::2] = rotated_x2
    return out

In [10]:
import torch
import torch.nn as nn
from torch.nn import functional as F
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
      super().__init__()
      self.head_size=head_size
      self.key = nn.Linear(n_embd, head_size, bias=False)
      self.query = nn.Linear(n_embd, head_size, bias=False)
      self.value = nn.Linear(n_embd, head_size, bias=False)
      cos, sin = precompute_rope_freqs(head_size, max_len, device)
      self.register_buffer('rope_cos', cos)
      self.register_buffer('rope_sin', sin)
      self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self,x):
      q=self.query(x)
      k=self.key(x)
      v=self.value(x)
      q = apply_rope(q, self.rope_cos, self.rope_sin)
      k = apply_rope(k, self.rope_cos, self.rope_sin)
      out = F.scaled_dot_product_attention(q, k, v, is_causal=True,dropout_p=0.1 if self.training else 0.0)
      B,T,C=x.shape
      return out

class Multihead(nn.Module):

  def __init__(self,no_heads,head_size):
    super().__init__()
    self.heads=nn.ModuleList([Head(head_size) for _ in range(no_heads)])
    self.projection= nn.Linear(no_heads*head_size,n_embd)

  def forward(self,x):
    outm=torch.cat([layer(x) for layer in self.heads],dim=-1)
    output=(self.projection)(outm)
    return output

class FeedForward(nn.Module):
  def __init__(self,dropout=0.1):
    super().__init__()
    self.layer=nn.Linear(n_embd,n_embd*4)
    self.layer2=nn.Linear(n_embd*4,n_embd)
    self.dropout = nn.Dropout(dropout)
    self.relu = nn.ReLU()

  def forward(self,x):
    init=(self.layer)(x)
    nonl=self.relu(init)
    out=(self.layer2)(nonl)
    out=self.dropout(out)
    return out

class Block(nn.Module):
  def __init__(self,n_embd,no_heads):
    super().__init__()
    head_size = n_embd // no_heads
    self.no_heads=no_heads
    self.ln1=nn.LayerNorm(n_embd)
    self.ln2=nn.LayerNorm(n_embd)
    self.ff=FeedForward()
    self.mh=Multihead(no_heads,head_size)


  def forward(self,x):
    out1=self.ln1(x)
    out2=self.mh(out1)
    x=x+out2
    out3=self.ln2(x)
    out4=self.ff(out3)
    x=x+out4
    return x

In [11]:
warmup_steps=200
max_steps=5000
max_lr=3e-4
min_lr=3e-5
losses_of_ropetrain=[]
losses_of_ropeval=[]
losses_of_postrain=[]
losses_of_posval=[]
iters_rope=[]
iters_pos=[]

In [12]:
experiment_name="ropedpolit_try3"
scaler = torch.amp.GradScaler('cuda')

In [13]:
import torch.nn as nn
from torch.nn import functional as F
class BigramLanguageModel(nn.Module):

  def __init__(self,no_layers):
    super().__init__()
    self.C=nn.Embedding(vocab_size,n_embd)

    self.finalnorm=nn.LayerNorm(n_embd)
    self.blocks = nn.Sequential(*[Block(n_embd, no_heads) for _ in range(no_layers)])
    self.lm_head=nn.Linear(n_embd,vocab_size)
    nn.init.normal_(self.C.weight, mean=0.0, std=0.02)
    self.lm_head.weight = self.C.weight



  def forward(self,x,target=None):
    emb1=self.C(x)
    B,T= x.shape
    finemb=emb1
    t=self.blocks(finemb)
    s=self.finalnorm(t)
    logits=self.lm_head(s)
    if target is None:
        loss = None
    else:
      B,T,C = logits.shape
      log=logits.view(B*T,vocab_size)
      target=target.view(B*T)
      loss = F.cross_entropy(log, target)

    return logits,loss

  def generate(self,idx,max_tokens):
    logits,loss=self(idx)
    for i in range(max_tokens):

      idx_cond=idx[:,-max_len:]
      logits,loss=self(idx_cond)
      logi=logits[:,-1,:]
      probs=torch.softmax(logi, dim=-1)

      idx_next = torch.multinomial(probs, num_samples=1)
      idx = torch.cat((idx, idx_next), dim=1)
    return idx

import math

def get_lr(step):

    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps

    elif step < max_steps:
        decay_ratio = (step - warmup_steps) / (max_steps - warmup_steps)
        coeff = 0.5 * (1 + math.cos(math.pi * decay_ratio))
        return min_lr + coeff * (max_lr - min_lr)

    else:
        return min_lr



model = BigramLanguageModel(no_layers=n_layer).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr)
print("warmup_steps:", warmup_steps, "max_steps:", max_steps, "max_lr:", max_lr, "min_lr:", min_lr)
print("n_embd:", n_embd, "block_size:", block_size, "no_heads:", no_heads, "n_layer:", n_layer)
print("lr at iter 0:", get_lr(0), "lr at iter 100:", get_lr(100))
max_iters = 5000



pretrained_path = f"{checkpoint_dir}/{experiment_name}.pt"

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(20)
        for k in range(20):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

if os.path.exists(pretrained_path):
    ckpt = torch.load(pretrained_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f"Loaded pretrained checkpoint from iter {ckpt['iter']}, skipping pretraining.")
else:
    for iter in range(max_iters):
        xb, yb = get_batch('train')
        lr = get_lr(iter)
        for param_group in optimizer.param_groups:
            param_group["lr"] = lr

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits, loss = model(xb, yb)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(optimizer)
        scaler.update()

        if iter % 100 == 0:
            print(f"iter {iter} running...")
            losses = estimate_loss()

            print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    torch.save({
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'iter': iter,
    }, pretrained_path)
    print("Pretrained checkpoint saved.")














warmup_steps: 200 max_steps: 5000 max_lr: 0.0003 min_lr: 3e-05
n_embd: 256 block_size: 128 no_heads: 8 n_layer: 8
lr at iter 0: 1.4999999999999998e-06 lr at iter 100: 0.0001515
iter 0 running...
step 0: train loss 10.8694, val loss 10.8794
iter 100 running...
step 100: train loss 7.8236, val loss 8.1879
iter 200 running...
step 200: train loss 5.9673, val loss 6.8820
iter 300 running...
step 300: train loss 5.2998, val loss 6.5577
iter 400 running...
step 400: train loss 5.0290, val loss 6.4016
iter 500 running...
step 500: train loss 4.8324, val loss 6.3566
iter 600 running...
step 600: train loss 4.6646, val loss 6.2642
iter 700 running...
step 700: train loss 4.5383, val loss 6.2389
iter 800 running...
step 800: train loss 4.4582, val loss 6.2662
iter 900 running...
step 900: train loss 4.3521, val loss 6.2416
iter 1000 running...
step 1000: train loss 4.2649, val loss 6.1407
iter 1100 running...
step 1100: train loss 4.2197, val loss 6.1470
iter 1200 running...
step 1200: train los

In [16]:
import copy
policy = model
ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad = False

In [17]:
val_text = enc.decode(val_data[:15000].tolist())
prompt_candidates = [val_text[i:i+30] for i in range( 0, 15000, 150)]

In [18]:
def score(model,context,gen_ids):
  context_len=len(context[0].tolist())
  full_ids = context[0].tolist() + gen_ids
  full_tensor = torch.tensor([full_ids], dtype=torch.long, device=device)
  x=full_tensor[:,:-1]
  y=full_tensor[:,context_len:]
  logits,_=model(x)
  probs=torch.log_softmax(logits[:,context_len-1:,:],-1)
  token_log_probs = probs.gather(-1, y.unsqueeze(-1)).squeeze(-1)
  avg_log_prob = token_log_probs.mean().item()
  return -avg_log_prob




def make_preference_pairs(ref_model, prompt_texts, num_samples=2, max_new_tokens=30):
    pairs = []
    ref_model.eval()
    with torch.no_grad():
        for prompt in prompt_texts:
            context = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
            completions_ids = []
            scores = []
            for _ in range(num_samples):
                out = ref_model.generate(context, max_new_tokens)
                gen_ids = out[0].tolist()[context.shape[1]:]
                completions_ids.append(gen_ids)
                scores.append(score(ref_model, context, gen_ids))

            best_idx = scores.index(min(scores))
            worst_idx = scores.index(max(scores))

            if best_idx != worst_idx:
                pairs.append({
                    "prompt": prompt,
                    "prompt_ids": context[0].tolist(),
                    "chosen_ids": completions_ids[best_idx],
                    "rejected_ids": completions_ids[worst_idx],
                })
    return pairs

In [19]:
def finding_logprobs(model, pairs, type):
    context_full = pairs["prompt_ids"] + pairs[type]
    context = torch.tensor(context_full, dtype=torch.long, device=device).unsqueeze(0)
    prompt_len = len(pairs["prompt_ids"])

    logits, _ = model(context[:, :-1])
    log_probs = torch.log_softmax(logits[:, prompt_len-1:, :], -1)
    targets = context[:, prompt_len:]

    token_log_probs = torch.gather(
        log_probs, dim=-1, index=targets.unsqueeze(-1)
    ).squeeze(-1)

    return token_log_probs.sum(dim=-1)



In [20]:
preference_pairs = make_preference_pairs(ref_model, prompt_candidates, num_samples=2, max_new_tokens=30)
beta = 0.1
num_epochs = 1

In [21]:
model.train()
ref_model.eval()

for epoch in range(num_epochs):
    for step, pairs in enumerate(preference_pairs):
        optimizer.zero_grad()

        policy_chosen_logprobs = finding_logprobs(model, pairs, "chosen_ids")
        policy_rejected_logprobs = finding_logprobs(model, pairs, "rejected_ids")

        with torch.no_grad():
            ref_chosen_logprobs = finding_logprobs(ref_model, pairs, "chosen_ids")
            ref_rejected_logprobs = finding_logprobs(ref_model, pairs, "rejected_ids")

        chosen_logratios = policy_chosen_logprobs - ref_chosen_logprobs
        rejected_logratios = policy_rejected_logprobs - ref_rejected_logprobs

        logits = chosen_logratios - rejected_logratios
        loss = -torch.nn.functional.logsigmoid(beta * logits).mean()

        reward_acc = (logits.detach() > 0).float().mean()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if step % 10 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f} reward_acc {reward_acc.item():.2f}")


torch.save({
    'model_state_dict': model.state_dict(),
}, f'{checkpoint_dir}/dpo_ckpt.pt')
print("DPO checkpoint saved.")

epoch 0 step 0 loss 0.5350 reward_acc 1.00
epoch 0 step 10 loss 0.5703 reward_acc 1.00
epoch 0 step 20 loss 0.4517 reward_acc 1.00
epoch 0 step 30 loss 0.4650 reward_acc 1.00
epoch 0 step 40 loss 0.4460 reward_acc 1.00
epoch 0 step 50 loss 0.0439 reward_acc 1.00
epoch 0 step 60 loss 0.6935 reward_acc 0.00
epoch 0 step 70 loss 0.0107 reward_acc 1.00
epoch 0 step 80 loss 0.0141 reward_acc 1.00
epoch 0 step 90 loss 0.0171 reward_acc 1.00
DPO checkpoint saved.


In [22]:
# =========================================================
# Append this after the DPO checkpoint save block.
# Generates side-by-side completions from the pretrained
# (ref_model) vs DPO-tuned (model) policy, scores them with
# your existing `score()` function, and writes a markdown
# table you can drop straight into your README.
# =========================================================

import textwrap

def clean(s):
    # collapse newlines so it renders inside a markdown table cell
    return s.replace("\n", " ").strip()

def generate_comparison(base_model, dpo_model, prompts, max_new_tokens=30, n_eval_prompts=8):
    """
    base_model: the frozen ref_model (pre-DPO)
    dpo_model:  the trained `model` (post-DPO)
    prompts:    list of raw prompt strings (held out — not used to build preference pairs)
    """
    base_model.eval()
    dpo_model.eval()

    rows = []
    with torch.no_grad():
        for prompt in prompts[:n_eval_prompts]:
            context = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)

            base_out = base_model.generate(context, max_new_tokens)
            dpo_out = dpo_model.generate(context, max_new_tokens)

            base_gen_ids = base_out[0].tolist()[context.shape[1]:]
            dpo_gen_ids = dpo_out[0].tolist()[context.shape[1]:]

            base_text = enc.decode(base_gen_ids)
            dpo_text = enc.decode(dpo_gen_ids)

            # lower score() = higher avg log-prob = model was more "confident"
            # in its own continuation, per your existing scoring function
            base_score = score(base_model, context, base_gen_ids)
            dpo_score = score(dpo_model, context, dpo_gen_ids)

            rows.append({
                "prompt": clean(prompt),
                "base": clean(base_text),
                "dpo": clean(dpo_text),
                "base_score": round(base_score, 3),
                "dpo_score": round(dpo_score, 3),
            })
    return rows


def rows_to_markdown_table(rows, max_cell_chars=120):
    def trunc(s):
        return s if len(s) <= max_cell_chars else s[:max_cell_chars].rstrip() + "…"

    lines = [
        "| Prompt | Base (pretrained) | DPO-tuned | Base score | DPO score |",
        "|---|---|---|---|---|",
    ]
    for r in rows:
        lines.append(
            f"| {trunc(r['prompt'])} | {trunc(r['base'])} | {trunc(r['dpo'])} "
            f"| {r['base_score']} | {r['dpo_score']} |"
        )
    return "\n".join(lines)


# Held-out prompts for eval — pull from val_data, further along than
# the slice used to build preference_pairs, so this isn't measuring
# something the policy was directly optimized on.
eval_prompt_texts = [ "sometimes life really is just",
    "i dont see the point in",
    "the thing about growing up is",
    "it is a truth universally acknowledged that",
    "curiouser and curiouser said",
    "it was the best of whales, it was the",
    "call me ishmael but honestly"]

comparison_rows = generate_comparison(ref_model, model, eval_prompt_texts, max_new_tokens=30)
markdown_table = rows_to_markdown_table(comparison_rows)

print(markdown_table)

readme_snippet_path = f"{checkpoint_dir}/dpo_comparison_table.md"
with open(readme_snippet_path, "w") as f:
    f.write("## Base vs. DPO-tuned completions\n\n")
    f.write(
        "Generations from the same pretrained checkpoint before and after DPO, "
        "on held-out prompts not used to build the preference pairs. "
        "Score is average negative log-likelihood the *generating* model assigns "
        "its own continuation (lower = more confident); it's a sanity signal, "
        "not a preference-quality metric on its own.\n\n"
    )
    f.write(markdown_table)
    f.write("\n")

print(f"\nSaved README-ready table to {readme_snippet_path}")

| Prompt | Base (pretrained) | DPO-tuned | Base score | DPO score |
|---|---|---|---|---|
| sometimes life really is just | that, to you who have masions in the secret, must be reserved in himself. You see I am a question of nature. At | .”  Miss Miller had to tell.  “Pretty,” I said; “so, I!” | 4.12 | 1.901 |
| i dont see the point in | the people’s sake, and the King sails--a pensive and so small their cook--left the fine giants as tilted | .”     oun’. I. .”  “Why, do you, that’s. | 4.132 | 1.491 |
| the thing about growing up is | the wash; but, by money, the steps  grabbed the smart thing and kill. He be jealous in every way or two, and | .’”  “I’d got to sit on’.”  “I’m | 4.554 | 1.209 |
| it is a truth universally acknowledged that | strong sum of condition would warm it. But if it would detach him for a little while to find comfort for you anywhere? I… | ..”  “Sir.”—“Youromes.”.—In a position.”  � | 3.979 | 1.994 |
| curiouser and curiouser said | nothing to him, but thought th